# Reentrenamiento por tipo de fraude

Para cada modelo (M1–M5) se entrenan **dos versiones**:
- `_tipo0`: positivos = fraudes del cluster 0, negativos = legítimas
- `_tipo1`: positivos = fraudes del cluster 1, negativos = legítimas

Los modelos re-entrenados heredan el mismo pipeline de preprocesamiento del original.
Solo se reemplaza el paso de clasificación con un estimador re-fitteado sobre los nuevos positivos.

**Input esperado**: un DataFrame `df_etiquetado` con las columnas del esquema Ánfora v6
más una columna `cluster_fraude` (0 o 1 para fraudes, NaN para legítimas).

**Output**: dict `modelos_retrained` con 10 entradas (5 modelos × 2 tipos).

## 0. Dependencias

In [16]:
import pickle, requests
import io
import numpy as np
import numpy.random as npr
import pandas as pd
from copy import deepcopy
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
import joblib, io, requests

warnings.filterwarnings('ignore')


SEED = 20260424
np.random.seed(SEED)

## 1. Descarga de modelos originales

In [17]:
MODEL_URLS = {
    "M1_logistica":    "https://anfora.netlify.app/datos/v6/modelos/M1_logistica.pkl",
    "M2_random_forest":"https://anfora.netlify.app/datos/v6/modelos/M2_random_forest.pkl",
    #"M3_gbm_completo": "https://anfora.netlify.app/datos/v6/modelos/M3_gbm_completo.pkl",
    #"M4_gbm_moderado": "https://anfora.netlify.app/datos/v6/modelos/M4_gbm_moderado.pkl",
    "M5_naive_bayes":  "https://anfora.netlify.app/datos/v6/modelos/M5_naive_bayes.pkl",
}

modelos_originales = {}
for nombre, url in MODEL_URLS.items():
    resp = requests.get(url)
    resp.raise_for_status()
    obj = joblib.load(io.BytesIO(resp.content))
    modelos_originales[nombre] = obj
    print(f"{nombre} — features: {obj['features']} — threshold: {obj['threshold']}")

M1_logistica — features: ['monto_log', 'canal', 'dispositivo'] — threshold: 0.14
M2_random_forest — features: ['monto_log', 'secuencia_24h', 'canal', 'dispositivo', 'hora_bin'] — threshold: 0.11
M5_naive_bayes — features: ['monto_log', 'secuencia_24h'] — threshold: 0.11


## 2. Carga de datos etiquetados con cluster

Reemplazá esta celda con tu fuente real.
`df_etiquetado` debe tener:
- todas las columnas del esquema Ánfora v6
- `es_fraude` (bool)
- `cluster_fraude` (0 o 1 para filas con `es_fraude == True`, NaN o -1 para legítimas)

In [18]:
# Primero el dataset de operaciones
df = pd.read_parquet("datos/v6/operaciones.parquet")

# Despues el que incluye los Cluster con los tipos de fraude
df_cluster = pd.read_parquet("datos/v6/clustering/clustering_operaciones.parquet")

# Hacemos un merge para tener toda la información en un mismo dataframe
df_merger = df.merge(df_cluster[["op_id", "cluster_fraude"]], on="op_id", how="left")

In [32]:
# Separamos en entrenamiento y test
df_etiquetado = df_merger[(df_merger["mes"] > 12)&(df_merger["mes"] <= 16)].copy()
df_test = df_merger[(df_merger["mes"] >= 17 )&(df_merger["mes"] <= 24)].copy()

In [20]:
# Chequeamos la distribución de fraudes y tipos de fraude en el dataset de entrenamiento

assert 'es_fraude' in df_etiquetado.columns
assert 'cluster_fraude' in df_etiquetado.columns

n_fraude = df_etiquetado['es_fraude'].sum()
n_legitimas = (~df_etiquetado['es_fraude']).sum()
n_tipo0 = (df_etiquetado['cluster_fraude'] == 0).sum()
n_tipo1 = (df_etiquetado['cluster_fraude'] == 1).sum()

print(f"Total registros : {len(df_etiquetado):,}")
print(f"Fraudes totales : {n_fraude:,}  ({n_fraude/len(df_etiquetado)*100:.2f}%)")
print(f"  cluster 0     : {n_tipo0:,}")
print(f"  cluster 1     : {n_tipo1:,}")
print(f"Legítimas       : {n_legitimas:,}")

Total registros : 201,442
Fraudes totales : 2,005  (1.00%)
  cluster 0     : 825
  cluster 1     : 1,180
Legítimas       : 199,437


## 3. Función de construcción del dataset por tipo

Para el tipo k:
- positivos: fraudes con `cluster_fraude == k`
- negativos: todas las legítimas

Los fraudes del otro cluster **se excluyen** del entrenamiento para que el modelo
aprenda únicamente la firma del tipo k sin contaminación.

In [21]:
def build_dataset(df: pd.DataFrame, tipo: int, features: list) -> tuple:
    """
    Construye X, y para entrenar el modelo del tipo `tipo`.
    
    Positivos : fraudes con cluster_fraude == tipo
    Negativos : legítimas (es_fraude == False)
    Excluidos : fraudes del otro cluster
    """
    mask_pos = df['cluster_fraude'] == tipo
    mask_neg = df['es_fraude'] == False
    
    df_train = pd.concat([
        df[mask_pos],
        df[mask_neg]
    ], axis=0).copy()
    
    # target binario: 1 = fraude de este tipo, 0 = legítima
    df_train['y'] = (df_train['cluster_fraude'] == tipo).astype(int)
    
    X = df_train[features]
    y = df_train['y']
    
    return X, y


print("Función build_dataset lista.")

Función build_dataset lista.


## 4. Función de reentrenamiento

El pipeline original tiene el preprocesamiento ya fiteado sobre T0.
Lo que hacemos es:
1. Hacer `deepcopy` del pipeline original (para no pisarlo)
2. Llamar `fit` sobre los nuevos X, y — sklearn re-fitea todo el pipeline incluyendo el clasificador

Nota: si el preprocesamiento incluye scalers o encoders, se van a re-fitear también sobre los
nuevos datos, lo cual es correcto: queremos que el preprocesamiento refleje la distribución de T1.

In [22]:
def retrain_model(modelo_obj: dict, df: pd.DataFrame, tipo: int) -> dict:
    """
    Re-entrena el modelo para detectar fraudes del `tipo` indicado.
    
    Devuelve un dict con las mismas keys que el original
    (pipeline, features, threshold) más:
      - 'tipo'        : int (0 o 1)
      - 'n_pos'       : cantidad de positivos usados
      - 'n_neg'       : cantidad de negativos usados
      - 'prior_pos'   : prevalencia de positivos en el train set
                        (necesaria luego para recuperar verosimilitud)
    """
    features   = modelo_obj['features']
    threshold  = modelo_obj['threshold']   # se hereda el umbral original
    pipeline   = deepcopy(modelo_obj['pipeline'])
    
    X, y = build_dataset(df, tipo, features)
    
    # re-fit completo
    pipeline.fit(X, y)
    
    n_pos = int(y.sum())
    n_neg = int((y == 0).sum())
    prior_pos = n_pos / (n_pos + n_neg)
    
    return {
        'pipeline'  : pipeline,
        'features'  : features,
        'threshold' : threshold,
        'tipo'      : tipo,
        'n_pos'     : n_pos,
        'n_neg'     : n_neg,
        'prior_pos' : prior_pos,
    }

print("Función retrain_model lista.")

Función retrain_model lista.


## 5. Reentrenamiento de los 5 modelos × 2 tipos

In [23]:
modelos_retrained = {}

for nombre, obj in modelos_originales.items():
    for tipo in [0, 1]:
        key = f"{nombre}_tipo{tipo}"
        print(f"Entrenando {key}...", end=' ')
        
        nuevo_modelo = retrain_model(obj, df_etiquetado, tipo)
        modelos_retrained[key] = nuevo_modelo
        
        print(f"OK  (positivos: {nuevo_modelo['n_pos']:,}  negativos: {nuevo_modelo['n_neg']:,}  "
              f"prior_pos: {nuevo_modelo['prior_pos']:.4f})")

print(f"\nTotal modelos entrenados: {len(modelos_retrained)}")

Entrenando M1_logistica_tipo0... OK  (positivos: 825  negativos: 199,437  prior_pos: 0.0041)
Entrenando M1_logistica_tipo1... OK  (positivos: 1,180  negativos: 199,437  prior_pos: 0.0059)
Entrenando M2_random_forest_tipo0... OK  (positivos: 825  negativos: 199,437  prior_pos: 0.0041)
Entrenando M2_random_forest_tipo1... OK  (positivos: 1,180  negativos: 199,437  prior_pos: 0.0059)
Entrenando M5_naive_bayes_tipo0... OK  (positivos: 825  negativos: 199,437  prior_pos: 0.0041)
Entrenando M5_naive_bayes_tipo1... OK  (positivos: 1,180  negativos: 199,437  prior_pos: 0.0059)

Total modelos entrenados: 6


## 6. Evaluación sobre el mismo conjunto de entrenamiento (sanity check)

Esto no reemplaza una evaluación sobre hold-out, pero sirve para verificar
que el pipeline fiteó correctamente y que las métricas son coherentes.

In [24]:
def evaluate_model(modelo_obj: dict, df: pd.DataFrame) -> dict:
    """Evalúa el modelo re-entrenado sobre el dataset de su tipo."""
    tipo      = modelo_obj['tipo']
    features  = modelo_obj['features']
    threshold = modelo_obj['threshold']
    pipeline  = modelo_obj['pipeline']
    
    X, y = build_dataset(df, tipo, features)
    
    proba = pipeline.predict_proba(X)[:, 1]
    pred  = (proba >= threshold).astype(int)
    
    return {
        'f1'        : f1_score(y, pred, zero_division=0),
        'precision' : precision_score(y, pred, zero_division=0),
        'recall'    : recall_score(y, pred, zero_division=0),
    }


rows = []
for key, obj in modelos_retrained.items():
    metrics = evaluate_model(obj, df_etiquetado)
    rows.append({
        'modelo': key,
        'tipo'  : obj['tipo'],
        'n_pos' : obj['n_pos'],
        **metrics
    })

df_eval = pd.DataFrame(rows).set_index('modelo')
print(df_eval.round(3).to_string())

                        tipo  n_pos     f1  precision  recall
modelo                                                       
M1_logistica_tipo0         0    825  0.203      0.260   0.166
M1_logistica_tipo1         1   1180  0.323      0.299   0.351
M2_random_forest_tipo0     0    825  0.356      0.395   0.325
M2_random_forest_tipo1     1   1180  0.427      0.373   0.500
M5_naive_bayes_tipo0       0    825  0.143      0.179   0.119
M5_naive_bayes_tipo1       1   1180  0.164      0.167   0.161


## 7. Inferencia sobre transacciones nuevas

### 7a. Posteriors y verosimilitudes

Para recuperar la **verosimilitud** P(x | tipo_k) a partir de la posterior
que devuelve `predict_proba`, corregimos por el prior de entrenamiento:

```
P(x | tipo_k)  ∝  P(tipo_k | x)  /  prior_pos_k
```

Esto elimina el sesgo de prevalencia absorbido durante el entrenamiento.

In [27]:
def score_transaccion(x_row: pd.DataFrame,
                      modelo_t0: dict,
                      modelo_t1: dict,
                      modelo_legitimas: dict = None,
                      w_tipo0: float = 0.5,
                      w_tipo1: float = 0.5) -> dict:
    """
    Calcula scores para una transacción nueva usando los modelos re-entrenados.

    Parámetros
    ----------
    x_row        : DataFrame de 1 fila con las features
    modelo_t0    : dict del modelo re-entrenado para tipo 0
    modelo_t1    : dict del modelo re-entrenado para tipo 1
    w_tipo0      : peso/prevalencia del tipo 0 en el mes actual (de Intelligence)
    w_tipo1      : peso/prevalencia del tipo 1 en el mes actual (de Intelligence)
                   w_tipo0 + w_tipo1 deben sumar 1

    Devuelve
    --------
    dict con:
      - post_tipo0   : P(tipo0 | x) — posterior cruda del modelo tipo 0
      - post_tipo1   : P(tipo1 | x) — posterior cruda del modelo tipo 1
      - like_tipo0   : verosimilitud proporcional P(x | tipo0)
      - like_tipo1   : verosimilitud proporcional P(x | tipo1)
      - like_fraude  : verosimilitud combinada P(x | H_f) = w0*like0 + w1*like1
      - surprise_t0  : -log P(x | tipo0)  (sorpresa bajo tipo 0)
      - surprise_t1  : -log P(x | tipo1)  (sorpresa bajo tipo 1)
      - alerta_tipo0 : bool — supera threshold del modelo tipo 0
      - alerta_tipo1 : bool — supera threshold del modelo tipo 1
      - alerta_any   : bool — alerta por cualquiera de los dos tipos
    """
    assert abs(w_tipo0 + w_tipo1 - 1.0) < 1e-6, "w_tipo0 + w_tipo1 debe ser 1"
    
    # features deben coincidir; usamos la intersección de las listas
    # (M3 tiene terminal_risk_score, el resto no)
    feat0 = modelo_t0['features']
    feat1 = modelo_t1['features']
    
    # posteriors
    post0 = modelo_t0['pipeline'].predict_proba(x_row[feat0])[:, 1][0]
    post1 = modelo_t1['pipeline'].predict_proba(x_row[feat1])[:, 1][0]
    
    # verosimilitudes proporcionales (corregidas por prior de entrenamiento)
    prior0 = modelo_t0['prior_pos']
    prior1 = modelo_t1['prior_pos']
    
    like0 = post0 / prior0
    like1 = post1 / prior1
    
    # LR
    LR = like1 / like0 if like0 > 0 else np.inf
    
    # verosimilitud combinada del fraude
    like_fraude = w_tipo0 * like0 + w_tipo1 * like1
    
    # sorpresa (surprise score): cuán inesperada es la transacción bajo cada tipo
    eps = 1e-10  # evitar log(0)
    surprise_t0 = -np.log(like0 + eps)
    surprise_t1 = -np.log(like1 + eps)
    
    # alertas por threshold
    alerta0 = post0 >= modelo_t0['threshold']
    alerta1 = post1 >= modelo_t1['threshold']
    
    return {
        'post_tipo0'      : round(post0, 6),
        'post_tipo1'      : round(post1, 6),
        'like_tipo0'      : round(like0, 6),
        'like_tipo1'      : round(like1, 6),
        'like_fraude'     : round(like_fraude, 6),
        'surprise_t0'     : round(surprise_t0, 4),
        'surprise_t1'     : round(surprise_t1, 4),
        'alerta_tipo0'    : bool(alerta0),
        'alerta_tipo1'    : bool(alerta1),
        'alerta_any'      : bool(alerta0 or alerta1),
        'LR(tipo1/tipo0)' : round(LR, 4)
    }

print("Función score_transaccion lista.")

Función score_transaccion lista.


In [33]:
# Podemos evaluar sobre transacciones puntuales del dataset de test (que no se usó para re-entrenar)
n = 1 # número de fila a evaluar 

score_transaccion(df_test.iloc[(n-1):n],modelo_t0= modelos_retrained['M2_random_forest_tipo0'], modelo_t1=modelos_retrained['M2_random_forest_tipo1'])

{'post_tipo0': 0.054248,
 'post_tipo1': 0.372596,
 'like_tipo0': 13.168218,
 'like_tipo1': 63.346712,
 'like_fraude': 38.257465,
 'surprise_t0': -2.5778,
 'surprise_t1': -4.1486,
 'alerta_tipo0': False,
 'alerta_tipo1': True,
 'alerta_any': True,
 'LR(tipo1/tipo0)': 4.8106}

In [34]:
# Resultado de la muestra de test:
df_test.iloc[(n-1):n]

,op_id,mes,canal,hora_bin,dispositivo,monto_log,terminal_risk_score,secuencia_24h,etiquetado,es_fraude,cluster_fraude
802734,802735,17,online,noche,desconocido,10.288115,0.238182,1,False,True,1.0


### 7b. Scoring en batch sobre un DataFrame completo

In [ ]:
def score_batch(df_nuevas: pd.DataFrame,
                modelos_retrained: dict,
                modelo_base: str = 'M4_gbm_moderado',
                w_tipo0: float = 0.5,
                w_tipo1: float = 0.5) -> pd.DataFrame:

    assert abs(w_tipo0 + w_tipo1 - 1.0) < 1e-6

    clave_t0 = f"{modelo_base}_tipo0"
    clave_t1 = f"{modelo_base}_tipo1"

    modelo_t0 = modelos_retrained[clave_t0]
    modelo_t1 = modelos_retrained[clave_t1]

    feat0 = modelo_t0['features']
    feat1 = modelo_t1['features']

    # Predicciones masivas
    post0 = modelo_t0['pipeline'].predict_proba(df_nuevas[feat0])[:, 1]
    post1 = modelo_t1['pipeline'].predict_proba(df_nuevas[feat1])[:, 1]

    prior0 = modelo_t0['prior_pos']
    prior1 = modelo_t1['prior_pos']

    like0 = post0 / prior0
    like1 = post1 / prior1

    # Deberían ser 2 LR? Uno para cada tipo de fraude contra no fraude? 
    LR = np.divide(
        like1,
        like0,
        out=np.full_like(like1, np.inf),
        where=like0 > 0
    )

    like_fraude = w_tipo0 * like0 + w_tipo1 * like1

    eps = 1e-10
    surprise_t0 = -np.log(like0 + eps)
    surprise_t1 = -np.log(like1 + eps)

    alerta0 = post0 >= modelo_t0['threshold']
    alerta1 = post1 >= modelo_t1['threshold']

    return pd.DataFrame({
        'post_tipo0': np.round(post0, 6),
        'post_tipo1': np.round(post1, 6),
        'like_tipo0': np.round(like0, 6),
        'like_tipo1': np.round(like1, 6),
        'like_fraude': np.round(like_fraude, 6),
        'surprise_t0': np.round(surprise_t0, 4),
        'surprise_t1': np.round(surprise_t1, 4),
        'alerta_tipo0': alerta0,
        'alerta_tipo1': alerta1,
        'alerta_any': alerta0 | alerta1,
        'LR(tipo1/tipo0)': np.round(LR, 4)
    }, index=df_nuevas.index)
    
print("Función score_batch lista.")

Función score_batch lista.


In [45]:
# Ejemplo de uso score_batch, es la generalización de score_transaccion para un dataset completo de nuevas transacciones.

# Pesos de Intelligence para el mes actual:
#   tipo 0 (fraude terminal, residual en T1) : ~20%
#   tipo 1 (smishing, dominante en T1)       : ~80%
#
df_mes_17 = df_test[df_test['mes'] == 17].copy()  # ejemplo con el mes 17 del dataset de test

scores = score_batch(
     df_mes_17,
     modelos_retrained,
     modelo_base='M2_random_forest',
     w_tipo0=0.2,
     w_tipo1=0.8
 )

print(f"Alertas totales: {scores['alerta_any'].sum()} ({scores['alerta_any'].mean()*100:.2f}%)")
scores

Alertas totales: 606 (1.25%)


,post_tipo0,post_tipo1,like_tipo0,like_tipo1,like_fraude,surprise_t0,surprise_t1,alerta_tipo0,alerta_tipo1,alerta_any,LR(tipo1/tipo0)
802734,0.054248,0.372596,13.168218,63.346712,53.311013,-2.5778,-4.1486,False,True,True,4.8106
802735,0.005344,0.012692,1.297214,2.157897,1.985760,-0.2602,-0.7691,False,False,False,1.6635
802736,0.000511,0.000137,0.124005,0.023343,0.043475,2.0874,3.7575,False,False,False,0.1882
802737,0.001269,0.000730,0.308056,0.124046,0.160848,1.1775,2.0871,False,False,False,0.4027
802738,0.007873,0.016158,1.911108,2.747162,2.579951,-0.6477,-1.0106,False,False,False,1.4375
...,...,...,...,...,...,...,...,...,...,...,...
851044,0.003934,0.005252,0.954845,0.892832,0.905235,0.0462,0.1134,False,False,False,0.9351
851045,0.000967,0.000199,0.234742,0.033852,0.074030,1.4493,3.3857,False,False,False,0.1442
851046,0.002044,0.003849,0.496277,0.654382,0.622761,0.7006,0.4241,False,False,False,1.3186
851047,0.000393,0.000171,0.095362,0.029144,0.042387,2.3501,3.5355,False,False,False,0.3056


## 8. Detección de régimen nuevo via surprise score

Una transacción con sorpresa alta bajo **ambos** tipos simultáneamente
no encaja con ningún patrón conocido. Es candidata a revisión manual
y potencialmente señal de un tercer régimen (T2 anticipado por Intelligence).

In [43]:
def detectar_anomalias(df_scores: pd.DataFrame,
                       umbral_surprise: float = 4.0) -> pd.DataFrame:
    """
    Marca transacciones con sorpresa alta en alguno de los tipos.
    
    umbral_surprise: en nats (4.0 = recomendación del reporte ene-feb 2025)
    """
    mask = (
        (df_scores['surprise_t0'] > umbral_surprise) |
        (df_scores['surprise_t1'] > umbral_surprise)
    )
    return df_scores[mask].copy()


# Ejemplo:
# anomalias = detectar_anomalias(scores, umbral_surprise=4.0)
# print(f"Transacciones de régimen desconocido: {len(anomalias)}")

print("Función detectar_anomalias lista.")

Función detectar_anomalias lista.


In [44]:
detectar_anomalias(scores, umbral_surprise=4.0)

,post_tipo0,post_tipo1,like_tipo0,like_tipo1,like_fraude,surprise_t0,surprise_t1,alerta_tipo0,alerta_tipo1,alerta_any,LR(tipo1/tipo0)
802780,0.000464,0.000103,0.112693,0.017473,0.026995,2.1831,4.0471,False,False,False,0.1550
802797,0.000468,0.000106,0.113653,0.017964,0.027533,2.1746,4.0194,False,False,False,0.1581
802844,0.000468,0.000106,0.113653,0.017964,0.027533,2.1746,4.0194,False,False,False,0.1581
803094,0.000468,0.000106,0.113653,0.017964,0.027533,2.1746,4.0194,False,False,False,0.1581
803200,0.000449,0.000098,0.108987,0.016680,0.025911,2.2165,4.0935,False,False,False,0.1530
...,...,...,...,...,...,...,...,...,...,...,...
1202014,0.000449,0.000100,0.108987,0.017072,0.026264,2.2165,4.0703,False,False,False,0.1566
1202088,0.000464,0.000106,0.112693,0.017995,0.027465,2.1831,4.0177,False,False,False,0.1597
1202319,0.000449,0.000100,0.108987,0.017072,0.026264,2.2165,4.0703,False,False,False,0.1566
1202587,0.000452,0.000103,0.109617,0.017473,0.026687,2.2108,4.0471,False,False,False,0.1594


## 9. Serialización de los modelos re-entrenados

In [33]:
import os

os.makedirs('modelos_retrained', exist_ok=True)

for key, obj in modelos_retrained.items():
    path = f'modelos_retrained/{key}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f"Guardado: {path}")

print("\nSerialización completa.")

Guardado: modelos_retrained/M1_logistica_tipo0.pkl
Guardado: modelos_retrained/M1_logistica_tipo1.pkl
Guardado: modelos_retrained/M2_random_forest_tipo0.pkl
Guardado: modelos_retrained/M2_random_forest_tipo1.pkl
Guardado: modelos_retrained/M5_naive_bayes_tipo0.pkl
Guardado: modelos_retrained/M5_naive_bayes_tipo1.pkl

Serialización completa.


## Resumen del pipeline

```
df_etiquetado (con cluster_fraude)
        │
        ├─ build_dataset(tipo=0) ──► retrain_model ──► M*_tipo0.pkl
        └─ build_dataset(tipo=1) ──► retrain_model ──► M*_tipo1.pkl

Transacción nueva
        │
        ├─ M*_tipo0.predict_proba → post0 → like0 = post0 / prior_pos0
        ├─ M*_tipo1.predict_proba → post1 → like1 = post1 / prior_pos1
        │
        ├─ like_fraude = w_tipo0 * like0 + w_tipo1 * like1
        │   (w de Intelligence, se actualiza mensualmente)
        │
        ├─ alerta si post_k >= threshold_k  (cualquiera de los dos)
        │
        └─ surprise_t0 alto Y surprise_t1 alto → régimen desconocido → revisión manual
```